In [1]:
import os
import joblib
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    r2_score,
    mean_squared_error,
    mean_absolute_error)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# --------------------------------------------------
# 1. MLflow Experiment
# --------------------------------------------------

In [2]:
mlflow.set_experiment("Planetary-Defense-Asteroid-Intelligence")

<Experiment: artifact_location='file:d:/Planetary Defense Analytics Project/mlruns/1', creation_time=1791726349128, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791726349128, lifecycle_stage='active', name='Planetary-Defense-Asteroid-Intelligence', tags={}, trace_location=None, workspace='default'>

# --------------------------------------------------
# CASE 1: PHA CLASSIFICATION
# --------------------------------------------------

In [3]:
# Load test data
X_test_c1 = pd.read_csv("processed_data/X_test.csv")
y_test_c1 = pd.read_csv("processed_data/y_test.csv").values.ravel()


In [4]:
# Load trained model
model_clf = joblib.load(
    "saved_models/best_case1_pha_classifier.joblib")

In [5]:
# Predictions
y_pred_c1 = model_clf.predict(X_test_c1)

# Classification metrics
accuracy = accuracy_score(y_test_c1, y_pred_c1)
precision = precision_score(y_test_c1, y_pred_c1, zero_division=0)
recall = recall_score(y_test_c1, y_pred_c1, zero_division=0)
f1 = f1_score(y_test_c1, y_pred_c1, zero_division=0)

In [ ]:
# Start MLflow run
with mlflow.start_run(run_name="Case1_PHA_XGBoost"):

    mlflow.log_param("task", "classification")
    mlflow.log_param("model_type", type(model_clf).__name__)

    # Log model hyperparameters when available
    if hasattr(model_clf, "get_params"):
        params = model_clf.get_params()
        for key in ["n_estimators", "max_depth", "learning_rate"]:
            if params.get(key) is not None:
                mlflow.log_param(key, params[key])

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

In [10]:

# ROC-AUC requires predicted probabilities
if hasattr(model_clf, "predict_proba"):
        y_proba_c1 = model_clf.predict_proba(X_test_c1)[:, 1]
        try:
            roc_auc = roc_auc_score(y_test_c1, y_proba_c1)
            mlflow.log_metric("roc_auc", roc_auc)
        except ValueError:
            print("ROC-AUC could not be calculated for these labels.")
 # Log the trained model
if type(model_clf).__module__.startswith("xgboost"):
        mlflow.xgboost.log_model(model_clf, name="classifier_model")
else:
        mlflow.sklearn.log_model(model_clf, name="classifier_model")

print("Case 1 metrics:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Case 1 metrics:
Accuracy: 0.9917163861069678
Precision: 0.18729463307776562
Recall: 0.8280871670702179
F1-score: 0.3054935238945958


# --------------------------------------------------
# CASE 2: DIAMETER REGRESSION
# --------------------------------------------------

In [11]:
print("\nStarting Case 2: Diameter Regression")


Starting Case 2: Diameter Regression


In [12]:
# Load test data
X_test_c2 = pd.read_csv("processed_regression/X_test.csv")
y_test_c2 = pd.read_csv("processed_regression/y_test.csv").values.ravel()


In [13]:
# Load trained model
model_reg = joblib.load(
    "saved_models/best_case2_diameter_regressor.joblib")


In [16]:
# Predict log-transformed diameter
y_pred_log_c2 = model_reg.predict(X_test_c2)

# Convert log1p values back to diameter in km
y_test_actual_c2 = np.expm1(y_test_c2)
y_pred_actual_c2 = np.expm1(y_pred_log_c2)

# Regression metrics on original km scale
r2 = r2_score(y_test_actual_c2, y_pred_actual_c2)
rmse = np.sqrt(mean_squared_error(y_test_actual_c2, y_pred_actual_c2))
mae = mean_absolute_error(y_test_actual_c2, y_pred_actual_c2)




In [20]:
with mlflow.start_run(run_name="Case2_Diameter_XGBoost"):

    mlflow.log_param("task", "regression")
    mlflow.log_param("model_type", type(model_reg).__name__)

    mlflow.log_param("target_transform", "log1p")
    mlflow.log_param("evaluation_scale", "diameter_km")

    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)

    mlflow.xgboost.log_model(model_reg,name="regressor_model")

print("Case 2 metrics:")
print("R²:", r2)
print("RMSE:", rmse, "km")
print("MAE:", mae, "km")

print("\nBoth models have been logged to MLflow.")    

print("Case 2 logged successfully!")

Case 2 metrics:
R²: 0.6012831098315177
RMSE: 6.342126599821458 km
MAE: 1.1142111419856322 km

Both models have been logged to MLflow.
Case 2 logged successfully!
